> **Research record.** Cleaned copy of `XXX-calculate-WFH/200-validate-remote-work-tickets.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Validates the remote-work metric against public-transport ticket validations.
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

In [ ]:
import pandas as pd
from geofunctions import utils
import geopandas as gpd

In [ ]:
df_2019 = pd.read_csv(
    utils.get_path("raw","statistics","transport_validations/tikufim_2019.csv"), sep= '|'
)

In [ ]:
df_2019['month_sum']= df_2019[df_2019.columns[-31:]].fillna(0).sum(axis=1)

In [ ]:
df_stat_list = []
for year in ['2019','2020','2021','2022','2023']:
    df_2019 = pd.read_csv(
        utils.get_path("raw","statistics",f"transport_validations/tikufim_{year}.csv"), sep= '|'
    )
    df_2019['month_sum']= df_2019[df_2019.columns[-31:]].fillna(0).sum(axis=1)
    df_2019_stat = df_2019.groupby(['month_key','year_key'])['month_sum'].sum().to_frame()
    df_stat_list.append(df_2019_stat)


In [ ]:
df_stat = pd.concat(df_stat_list)

In [ ]:
df_stat =df_stat.reset_index()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Ensure df_stat contains the correct columns
df_stat['date'] = pd.to_datetime(df_stat['year_key'].astype(str) + '-' + df_stat['month_key'].astype(str))
df_stat.set_index('date', inplace=True)

# Sort values to ensure proper chronological order
df_stat.sort_index(inplace=True)

# Plot
plt.figure(figsize=(12, 6))
plt.plot(df_stat.index, df_stat['month_sum'], marker='o', linestyle='-', linewidth=2, markersize=5, label='Monthly Sum')

# Formatting the plot
plt.xlabel("Date")
plt.ylabel("Value")
plt.title("Time-Series Plot of Monthly Sum")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Show plot
plt.show()


In [ ]:
df_stat['month'] = df_stat['year_key'].astype(str)+df_stat['month_key'].astype(str)

In [ ]:
# Convert df1['month'] to YYYY-MM format
df_stat['month'] = df_stat.reset_index()['date'].dt.strftime('%Y-%m').values



## Add remote work data

In [ ]:
df_rw = pd.read_csv("../data/diagrams_data/remote_work_statistic.csv")

In [ ]:
# Convert df2['month'] to YYYY-MM format
df_rw['month'] = df_rw['month'].astype(str).str[:4] + '-' + df_rw['month'].astype(str).str[4:]


In [ ]:
df_merged = df_stat.merge(df_rw, on = 'month')

In [ ]:
df_merged['reverse_remote_work'] = 1-df_merged['flag_remote_work_smoothed_max_thresh']

In [ ]:
df_merged['date'] = df_merged['month']

In [ ]:
df_merged['month_sum_norm'] = df_merged['month_sum']/79811782.0

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Ensure 'month' column is datetime and set as index
df_merged['month'] = pd.to_datetime(df_merged['month'])
df_merged.set_index('month', inplace=True)

# Create figure and axis
fig, ax1 = plt.subplots(figsize=(12, 6))

# First Y-axis (Primary)
color1 = 'tab:blue'
ax1.set_xlabel('Month')
ax1.set_ylabel('Flag Remote Work', color=color1)
ax1.plot(df_merged.index, df_merged['reverse_remote_work'], 
         color=color1, marker='o', linestyle='-', label='Inverse Remote Work')
ax1.tick_params(axis='y', labelcolor=color1)

# Format X-axis to show 'YYYY-MM'
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))  # Show every 3 months
plt.xticks(rotation=90)

# Second Y-axis (Secondary)
ax2 = ax1.twinx()  
color2 = 'tab:red'
ax2.set_ylabel('Month Sum', color=color2)
ax2.plot(df_merged.index, df_merged['month_sum_norm'], 
         color=color2, marker='s', linestyle='--', label='Month Sum')
ax2.tick_params(axis='y', labelcolor=color2)

# Title and grid
plt.title("Comparison of Inverse Remote Work & Monthly Sum Over Time")
fig.tight_layout()
plt.grid(True, linestyle='--', alpha=0.6)

# Show plot
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Ensure 'month' column is datetime and set as index


# Create figure and axis
fig, ax = plt.subplots(figsize=(12, 6))

# Plot both lines on the same y-axis
ax.plot(df_merged.index, df_merged['reverse_remote_work'], 
        color='tab:blue', marker='o', linestyle='-', label='Inverse Remote Work')

ax.plot(df_merged.index, df_merged['month_sum_norm'], 
        color='tab:red', marker='s', linestyle='--', label='Month Sum')

# Labels and formatting
ax.set_xlabel("Month")
ax.set_ylabel("Value")
ax.set_title("Comparison of Inverse Remote Work & Monthly Sum Over Time")

# Format X-axis to show 'YYYY-MM'
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))  # Show every 3 months
plt.xticks(rotation=90)

# Add legend and grid
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)

# Show plot
plt.show()
